# 歡迎來到內容豐富的第 8 週資料夾

## 本週任務很多！

我們會比平常更快，尤其你已成為熟練的 LLM 工程師。

# 猜價格（The Price is Right）

## 第 8 週進行順序

第 1 天：Modal.com 與 SpecialistAgent  
第 2 天：RAG、FrontierAgent、Ensemble Agent  
第 3 天：ScannerAgent、MessengerAgent  
第 4 天：AutonomousPlannerAgent 與 DealAgentFramework  
第 5 天：猜價格總決賽


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#b22;">本週特別重要：拉取最新程式碼</h2>
            <span style="color:#b22;">我持續改進這些 lab，新增範例與練習。
            每週開始時，值得確認你已有最新程式碼。<br/>
            若不確定如何 <code>git pull</code>，請見 Guides 資料夾的 Guide 3
            </span>
        </td>
    </tr>
</table>

In [ ]:
import os
import locale
import modal
from agents.preprocessor import Preprocessor
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
# 檢查電腦能否正確輸出特殊字元，確認此處輸出為 UTF-8

print(locale.getpreferredencoding())  # Should print 'UTF-8'

In [ ]:
os.environ["PYTHONIOENCODING"] = "utf-8"

# 設定 modal tokens

## 重要——請閱讀並遵循以下說明！

首先造訪：https://modal.com

註冊帳號。點右上角 Avatar 選單，選「Settings」

左側點「API Tokens」，再點「New Token」。

你會得到類似以下的指令：

`modal token set --token-id ak-somethinghere --token-secret as-somethinghere`

因為我們用 uv，實際要執行的是：

`uv run modal token set --token-id ak-somethinghere --token-secret as-somethinghere`

### 疑難排解

若有問題，可試三件事：

1. 在 `uv run modal token set..` 之前先執行 `uv run modal token new`  

2. Windows 學員 David S. 的建議：

> 若其他 Windows 使用者遇到同樣問題：除了要在命令提示字元執行 `modal token new`，還得移動產生的 token 檔。它會把 .modal.toml 放到 Windows 使用者設定檔資料夾。虛擬環境看不到該位置（奇怪的是，設了環境變數並重開機後仍不行）。我把 token 檔移到 lab 工作目錄後，就不再出現 auth 錯誤。

3. 手動方式：

也可直接把兩個 key 加入 .env：

```
MODAL_TOKEN_ID=ak-...
MODAL_TOKEN_SECRET=as-...
```

然後重新執行 `load_dotenv(override=True)` 載入環境變數。

In [ ]:
from hello import app, hello, hello_europe

In [ ]:
with app.run():
    reply=hello.local()
reply

In [ ]:
with app.run():
    reply=hello.remote()
reply

## 感謝學員 Tue H. 補充

查看 hello.py，我加了簡單函式 hello_europe

使用 decorator：  
`@app.function(image=image, region="eu")`

見下方結果！更多區域設定 [在此](https://modal.com/docs/guide/region-selection)

注意：指定區域會稍微多消耗 credits。

In [ ]:
with app.run():
    reply=hello_europe.remote()
reply

# 繼續之前——

## 我們要在 Modal 中把你的 HuggingFace Token 設為 secret

## 超重要——請閱讀——這常讓人困惑！

Modal 的 secret 有描述性的 **名稱**。  
secret 本身有 KEY 與 VALUE。  
我們將設定：  

Name: huggingface-secret  
Key: HF_TOKEN  
Value: hf_...  

## 萬無一失的步驟：

1. 前往 modal.com，登入並進入 dashboard  
2. 導覽列點 Secrets  
3. 建立新 secret，點 Hugging Face，此 secret 必須叫 **huggingface-secret**，因為程式碼這樣引用  
4. Key 填 HF_TOKEN，Value 填你的 token hf_...  
5. 點 done

### 回到正題：是時候用 Llama 了

In [ ]:
# 此 import 可能出現關於將本機 Python 模組加入 Image 的棄用警告
# 可安全忽略。其他地方也可能出現相同警告……

from llama import app, generate

In [ ]:
with modal.enable_output():
    with app.run():
        result=generate.remote("Never gonna give you up, never gonna")
result

# 若不想被 rickroll，可試："Hey Jude, don't make it"

In [ ]:
from pricer_ephemeral import app, price

In [ ]:
with modal.enable_output():
    with app.run():
        result=price.remote("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
result

In [ ]:
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

In [ ]:
preprocessor = Preprocessor(model_name="groq/openai/gpt-oss-20b")
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

### 若希望 Preprocessor 預設使用不同模型，在 .env 加入：

`PRICER_PREPROCESSOR_MODEL=groq/openai/gpt-oss-20b`

In [ ]:
with modal.enable_output():
    with app.run():
        result = price.remote(text)
print(result)

In [ ]:
print(text)

## 從 Ephemeral Apps 過渡到 Deployed Apps

在命令列，`uv run modal deploy xxx` 會把程式碼部署為 Deployed App

這是把 AI 服務包成 API、用於 Production 系統的方式。

也可輕鬆建 REST endpoint，但本課程不涵蓋，我們會直接從 Python 呼叫。

## 關於 secrets 的重要說明

在 `pricer_service.py` 與 `pricer_service2.py` 頂部附近有類似：  
`secrets = [modal.Secret.from_name("hf-secret")]`  
視 Modal 設定，可能需把 `hf-secret` 改成 `huggingface-secret`。  
請至下列頁面看第一欄確認：  
https://modal.com/secrets/

## Windows 使用者注意：

下一行我在 Jupyter lab 內呼叫 `uv run modal deploy`；聽說部分 Windows 版本會因 modal 輸出 emoji 無法顯示而出現 unicode 錯誤。若發生，請開 Terminal 執行 `uv run modal deploy..`

In [ ]:
# 也可在 Terminal 執行 "uv run modal deploy -m pricer_service"

!uv run modal deploy -m pricer_service

In [ ]:
pricer = modal.Function.from_name("pricer-service", "price")

觀看部署過程：

https://modal.com

In [ ]:
# 這可能需要一段時間！我們很快會用更快的方法

pricer.remote(text)

In [ ]:
# 也可在已啟動的環境命令列執行 "modal deploy -m pricer_service2"

!modal deploy -m pricer_service2

In [ ]:
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
reply = pricer.price.remote(text)
print(reply)

In [ ]:
reply = pricer.price.remote(text)
print(reply)

# 選用：讓 Modal 保持 warm

## 提升 Modal pricer 服務速度的方法

首次執行此 modal class 可能需長達 10 分鐘 build。  
之後應快得多……若需喚醒約 30 秒，否則約 2 秒。  
若希望永遠 2 秒，可在 pricer_service2.py 編輯常數，避免 container 休眠：

`MIN_CONTAINERS = 0`



改為 1 可保持一個 container 存活。  
但請注意：會持續消耗 credits！僅在你能接受常駐程序時這麼做。

或者執行以下程式碼，可保持 warm 20 分鐘而非 2 分鐘。

### 保持 warm 20 分鐘再冷卻的程式碼：

```python
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=1200)
```

### 恢復僅保持 warm 2 分鐘的程式碼

```python
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=120)
```

## 介紹我們的 Agent class

預設會用 Llama3.2 做 preprocess

若偏好 Groq，加入環境變數：

```
PRICER_PREPROCESSOR_MODEL=groq/openai/gpt-oss-20b
```


In [ ]:
import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.specialist_agent import SpecialistAgent

In [ ]:
agent = SpecialistAgent()


In [ ]:
agent.price("iPhone 10")